# ROSALIA on PadChest-GR — uncertainty-stratified IoU eval (two-phase)

For every (image, grounded finding) pair in PadChest-GR whose finding maps to
one of ROSALIA's **7 supported lesion concepts**
(`cardiomegaly, pneumonia, atelectasis, opacity, consolidation, edema, effusion`),
we run ROSALIA (LISA-7B + SAM-H fine-tuned on 1.1M MIMIC-ILS pairs) to get a
predicted lesion mask, then compute a bundle of IoUs vs the two annotators'
ground-truth boxes, aggregated by the rule-based spatial-uncertainty label.

## ⚠️ This notebook has TWO phases separated by a runtime restart

**Why:** MedGemma-4b-it is a **Gemma-3** model that needs `transformers>=4.50`,
but ROSALIA/LISA needs **exactly** `transformers==4.31.0`. The two cannot live
in the same runtime. So:

* **Phase A — MedGemma classifier.** Uses MedGemma to map every *unique*
  PadChest-GR `finding_label` to one of the 7 ROSALIA concepts (or `none`,
  which is dropped). Runs once, saves `finding_concept_map.json` to `/content`
  and GCS.
* **↻ Restart runtime + reinstall.**
* **Phase B — ROSALIA.** Loads the cached map, runs segmentation + IoU eval.

**Run order:** Phase A cells → Cell B1 (installs ROSALIA deps) → **Restart
runtime** → Phase B cells (B2 onward).

**Runtime:** GPU with ≥16 GB VRAM (L4/A100 recommended; ROSALIA-7B bf16 is
~15 GB, borderline OOM on a T4). MedGemma-4b needs only ~9 GB, so Phase A is
fine on any GPU.


## Cell 0 — one-time GCS extraction of the split-zip PadChest-GR archive

Idempotent: if `gs://${BUCKET}/images/` is already populated, it no-ops. You
can run this in either phase (it doesn't touch transformers).


In [ ]:
BUCKET      = "yang-padchest-gr"
GCS_PROJECT = "yang-uncertainty-eval"
SRC_PREFIX  = "Padchest_GR_files"
DST_PREFIX  = "images"

from google.colab import auth
auth.authenticate_user()
get_ipython().system(f"gcloud config set project {GCS_PROJECT} -q")
print("Active identity:")
get_ipython().system("gcloud auth list --filter=status:ACTIVE --format='value(account)'")

import subprocess
r = subprocess.run(
    f"gcloud storage ls gs://{BUCKET}/ 2>&1 | head -10",
    capture_output=True, text=True, shell=True,
)
print(f"\nPreflight `gcloud storage ls gs://{BUCKET}/`:")
print(r.stdout)
if "ERROR" in r.stdout or "403" in r.stdout or "404" in r.stdout or "401" in r.stdout:
    raise RuntimeError(
        f"Cannot read gs://{BUCKET}/ as the active account. Check bucket name, "
        f"signed-in account, and project '{GCS_PROJECT}'."
    )

from google.cloud import storage
_client = storage.Client(project=GCS_PROJECT)
_bucket = _client.bucket(BUCKET)

def gcs_has_images():
    return len(list(_bucket.list_blobs(prefix=f"{DST_PREFIX}/", max_results=1))) > 0

if gcs_has_images():
    print(f"\ngs://{BUCKET}/{DST_PREFIX}/ already populated — skipping extraction.")
else:
    print("\nExtracting PadChest-GR from split-zip parts. One-time slow step.")
    !mkdir -p /content/zips /content/images /content/extracted
    get_ipython().system(
        f"gcloud storage cp 'gs://{BUCKET}/{SRC_PREFIX}/PadChest_GR.zip.*' /content/zips/")
    !ls /content/zips/PadChest_GR.zip.* | sort > /content/zip_parts.txt
    !cat $(cat /content/zip_parts.txt | tr '\n' ' ') > /content/full.zip
    !rm -f /content/zips/PadChest_GR.zip.*
    !unzip -q -o /content/full.zip -d /content/extracted/
    !rm -f /content/full.zip
    !find /content/extracted -type f -name '*.png' -print0 | xargs -0 -I{} cp {} /content/images/
    get_ipython().system(
        f"gcloud storage rsync /content/images/ gs://{BUCKET}/{DST_PREFIX}/ --recursive")
    !rm -rf /content/extracted /content/images
    print("Extraction + upload complete.")


# ══════════════════ PHASE A — MedGemma finding classifier ══════════════════

Run Cells A1 → A3 in a **fresh runtime with modern transformers** (Colab's
default). This produces `finding_concept_map.json`. When A3 finishes, go to
Cell B1.


## Cell A1 — install MedGemma deps (modern transformers)

MedGemma-4b-it is Gemma-3 (`image-text-to-text`) and needs `transformers>=4.50`
and a recent `accelerate`. Colab usually has these, but we pin to be safe.


In [ ]:
!pip -q install --upgrade 'transformers>=4.50' 'accelerate>=0.30' gcsfs google-cloud-storage huggingface_hub pandas

import transformers, torch
print("transformers", transformers.__version__)
print("torch       ", torch.__version__)
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 50), \
    "Need transformers>=4.50 for MedGemma (Gemma-3). Restart runtime if it just upgraded."


## Cell A2 — authenticate (Google + Hugging Face)

* Google auth → save the map to GCS.
* HF login → `google/medgemma-4b-it` is a **gated** model; you must have
  accepted its license at https://huggingface.co/google/medgemma-4b-it and log
  in with a token that has access.


In [ ]:
from google.colab import auth
auth.authenticate_user()

from huggingface_hub import notebook_login
notebook_login()


## Cell A3 — classify each unique finding label with MedGemma

1. Download the samples CSV from GitHub.
2. Collect the **unique** `finding_label`s + one representative sentence each
   (so MedGemma sees context, e.g. "opacity consistent with pneumonia").
3. Ask MedGemma to pick **exactly one** of
   `{cardiomegaly, pneumonia, atelectasis, opacity, consolidation, edema,
   effusion, none}`. `none` = "not one of ROSALIA's 7 → will be dropped".
   Few-shot examples anchor the behavior; parsing is strict (invalid →
   `none`, the safe default).
4. Save the mapping to `/content/finding_concept_map.json` and
   `gs://${BUCKET}/outputs/rosalia_v1/finding_concept_map.json` so it survives
   the Phase-B runtime restart.
5. Print a review table so you can eyeball MedGemma's choices.

This runs once over ~100-200 unique labels (not per sample), so it's fast.


In [ ]:
import os, json, re, gc
import pandas as pd
import torch

BUCKET       = "yang-padchest-gr"
GCS_PROJECT  = "yang-uncertainty-eval"
MAP_GCS_PATH = f"{BUCKET}/outputs/rosalia_v1/finding_concept_map.json"
MAP_LOCAL    = "/content/finding_concept_map.json"

REPO_RAW_URL = "https://raw.githubusercontent.com/nprakash1/uncertaintyestimateyang/rule-bag-of-words"
SAMPLES_CSV  = "/content/samples_with_uncertainty_and_iou.csv"
if not os.path.exists(SAMPLES_CSV):
    get_ipython().system(
        f"curl -fsSL '{REPO_RAW_URL}/project/data/processed/samples_with_uncertainty_and_iou.csv' -o '{SAMPLES_CSV}'")

samples = pd.read_csv(SAMPLES_CSV)
print(f"Loaded {len(samples)} rows; {samples['finding_label'].nunique()} unique finding labels")

# One representative sentence per unique label (first non-null occurrence)
rep = (samples.dropna(subset=["finding_label"])
              .groupby("finding_label")["sentence"]
              .first().to_dict())
unique_labels = sorted(rep.keys())

ROSALIA_LESIONS = ["cardiomegaly","pneumonia","atelectasis","opacity",
                   "consolidation","edema","effusion"]
ALLOWED = set(ROSALIA_LESIONS) | {"none"}

FEWSHOT = (
    "Examples:\n"
    "Finding: 'pleural effusion' | Sentence: 'blunting of the costophrenic angle' -> effusion\n"
    "Finding: 'cardiomegaly' | Sentence: 'the cardiac silhouette is enlarged' -> cardiomegaly\n"
    "Finding: 'alveolar pattern' | Sentence: 'alveolar opacities in the right base' -> opacity\n"
    "Finding: 'laminar atelectasis' | Sentence: 'linear atelectasis at left base' -> atelectasis\n"
    "Finding: 'pulmonary edema' | Sentence: 'signs of interstitial edema' -> edema\n"
    "Finding: 'consolidation' | Sentence: 'dense consolidation in right lower lobe' -> consolidation\n"
    "Finding: 'pulmonary nodule' | Sentence: 'a solitary nodule is seen' -> none\n"
    "Finding: 'pneumothorax' | Sentence: 'apical pneumothorax' -> none\n"
    "Finding: 'aortic elongation' | Sentence: 'unfolded thoracic aorta' -> none\n"
)

def build_prompt(label, sentence):
    return (
        "You are a radiology label normalizer. A downstream segmentation model "
        "can ONLY segment these 7 chest X-ray lesion types:\n"
        f"{', '.join(ROSALIA_LESIONS)}.\n\n"
        "Given a PadChest-GR finding label and an example sentence, respond with "
        "EXACTLY ONE word: the single best-matching lesion type from that list, "
        "or 'none' if the finding is not one of those 7 (e.g. nodule, mass, "
        "pneumothorax, fracture, aortic changes, hardware/devices, normal "
        "anatomy). Output only one lowercase word, no punctuation.\n\n"
        f"{FEWSHOT}\n"
        f"Finding: '{label}' | Sentence: '{str(sentence)[:200]}' ->"
    )

# --- load MedGemma (Gemma-3 image-text-to-text; needs transformers>=4.50) ---
from transformers import AutoProcessor, AutoModelForImageTextToText
MODEL_NAME = "google/medgemma-4b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} on {device} ...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
tok = getattr(processor, "tokenizer", None) or processor
tok.padding_side = "left"
if getattr(tok, "pad_token", None) is None and getattr(tok, "eos_token", None):
    tok.pad_token = tok.eos_token
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()

def classify(label, sentence):
    prompt = build_prompt(label, sentence)
    msgs = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=6, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    raw = tok.batch_decode(gen, skip_special_tokens=True)[0]
    word = re.sub(r"[^a-z]", "", raw.strip().lower().split()[0]) if raw.strip() else "none"
    return word if word in ALLOWED else "none", raw.strip()

from tqdm.auto import tqdm
finding_concept_map = {}
raw_outputs = {}
for lbl in tqdm(unique_labels, desc="MedGemma classify"):
    concept, raw = classify(lbl, rep[lbl])
    finding_concept_map[lbl] = concept
    raw_outputs[lbl] = raw

# --- review table ---
review = (pd.DataFrame({
            "finding_label": list(finding_concept_map),
            "concept": list(finding_concept_map.values()),
            "medgemma_raw": [raw_outputs[k] for k in finding_concept_map],
            "n_samples": [int((samples["finding_label"]==k).sum()) for k in finding_concept_map],
         })
         .sort_values(["concept","n_samples"], ascending=[True, False]))
print("\n=== MedGemma finding -> ROSALIA concept mapping ===")
import pandas as _pd; _pd.set_option("display.max_rows", 300)
print(review.to_string(index=False))

n_kept = int((review["concept"]!="none").sum())
kept_samples = int(samples["finding_label"].map(finding_concept_map).isin(ROSALIA_LESIONS).sum())
print(f"\nUnique labels: {len(unique_labels)} | mapped to a concept: {n_kept} | 'none': {len(unique_labels)-n_kept}")
print(f"Samples kept (in-vocab): {kept_samples} / {len(samples)} ({100*kept_samples/len(samples):.1f}%)")
print("\nConcept distribution (by sample count):")
print(samples["finding_label"].map(finding_concept_map).value_counts().to_string())

# --- persist to /content + GCS (survives the Phase-B restart) ---
with open(MAP_LOCAL, "w") as f:
    json.dump(finding_concept_map, f, indent=2)
print(f"\nSaved {MAP_LOCAL}")
try:
    import gcsfs
    fs = gcsfs.GCSFileSystem(project=GCS_PROJECT)
    with fs.open(MAP_GCS_PATH, "w") as f:
        f.write(json.dumps(finding_concept_map, indent=2))
    print(f"Saved gs://{MAP_GCS_PATH}")
except Exception as e:
    print(f"[warn] GCS save failed ({e}); the /content copy will still work if you don't delete the runtime.")

# --- free MedGemma VRAM before Phase B ---
del model, processor
gc.collect(); torch.cuda.empty_cache()
print("\nMedGemma unloaded. Proceed to Cell B1.")


# ══════════════════ PHASE B — ROSALIA segmentation ══════════════════

Cell B1 reinstalls the ROSALIA dependency stack (which **downgrades**
transformers to 4.31.0 and is incompatible with MedGemma). After B1 you MUST
**restart the runtime**, then run B2 onward.


## Cell B1 — install ROSALIA deps + clone repo, then RESTART

`transformers==4.31.0` is required by LISA's custom `LISAForCausalLM`. This
will uninstall the MedGemma-compatible transformers — that's expected. The
`finding_concept_map.json` from Phase A is already saved to disk + GCS, so it
survives the restart.


In [ ]:
# CUDA-12.6 PyTorch wheels (LISA/ROSALIA compatible)
!pip -q install --force-reinstall --no-deps torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu126

import os
if not os.path.isdir('/content/rosalia_repo'):
    !git clone https://github.com/checkoneee/ROSALIA.git /content/rosalia_repo

# ROSALIA pinned deps — transformers==4.31.0 is CRITICAL.
!pip -q install 'transformers==4.31.0' 'peft==0.4.0' 'einops==0.4.1' \
    'sentencepiece' 'opencv-python==4.8.0.74' 'pycocotools>=2.0.7' \
    'scikit-image' 'bitsandbytes==0.48.2'
!pip -q install --upgrade gcsfs google-cloud-storage huggingface_hub

import subprocess
subprocess.run(["python","-c",
                "import transformers,peft,einops,torch;"
                "print('transformers',transformers.__version__);"
                "print('peft',peft.__version__);print('torch',torch.__version__)"])
print("\n>>> RESTART THE RUNTIME NOW, then run Cell B2 onward. <<<")


## Cell B2 — authenticate (Google + Hugging Face)

`checkone/ROSALIA-7B-v1` and `xinlai/LISA-7B-v1` are public/ungated; auth just
raises rate limits and enables GCS access.


In [ ]:
from google.colab import auth
auth.authenticate_user()
from huggingface_hub import notebook_login
notebook_login()


## Cell B3 — config

In [ ]:
GCS_PROJECT = "yang-uncertainty-eval"
BUCKET      = "yang-padchest-gr"
IMG_PREFIX  = "images"
OUT_PREFIX  = "outputs/rosalia_v1"
MASK_PREFIX = f"{OUT_PREFIX}/masks"
MAP_GCS_PATH = f"{BUCKET}/{OUT_PREFIX}/finding_concept_map.json"
MAP_LOCAL    = "/content/finding_concept_map.json"

DRIVE_REPO_ROOT = "/content/drive/MyDrive/uncertaintyestimate"

ROSALIA_REPO      = "checkone/ROSALIA-7B-v1"
LISA_TOKENIZER    = "xinlai/LISA-7B-v1"
CLIP_VISION_TOWER = "openai/clip-vit-large-patch14"

# Instruction template: "global" = "Segment the [target]." (no location parse)
INSTRUCTION_MODE       = "global"
INSTRUCTION_TEMPLATE_G = "Segment the {target}."
INSTRUCTION_TEMPLATE_B = "Segment the {target} in the {location}."

ROSALIA_LESIONS = {"cardiomegaly","pneumonia","atelectasis","opacity",
                   "consolidation","edema","effusion"}

MASK_GRID   = 1024
MASK_THRESHOLD = 0.5
SMOKE_N = 50
SMOKE_SEED = 42

print("Config loaded. Instruction mode:", INSTRUCTION_MODE)


## Cell B4 — load samples + rule labels, apply MedGemma map, filter to vocab

Loads the cached `finding_concept_map.json` (from Phase A) — first from
`/content`, else from GCS. Applies it, drops `none`, and builds the ROSALIA
instruction per row. If the map is missing, re-run Phase A.


In [ ]:
import os, json, pandas as pd
import gcsfs
fs = gcsfs.GCSFileSystem(project=GCS_PROJECT)

REPO_RAW_URL = "https://raw.githubusercontent.com/nprakash1/uncertaintyestimateyang/rule-bag-of-words"
SAMPLES_CSV  = "/content/samples_with_uncertainty_and_iou.csv"
RULE_JSONL   = "/content/rule_uncertainty_scores.jsonl"
for fn, url in [
    (SAMPLES_CSV, f"{REPO_RAW_URL}/project/data/processed/samples_with_uncertainty_and_iou.csv"),
    (RULE_JSONL,  f"{REPO_RAW_URL}/project/data/processed/rule_uncertainty_scores.jsonl"),
]:
    if not os.path.exists(fn):
        get_ipython().system(f"curl -fsSL '{url}' -o '{fn}'")

# --- load the MedGemma-produced map ---
finding_concept_map = None
if os.path.exists(MAP_LOCAL):
    finding_concept_map = json.load(open(MAP_LOCAL))
    print(f"Loaded map from {MAP_LOCAL} ({len(finding_concept_map)} labels)")
else:
    try:
        with fs.open(MAP_GCS_PATH, "r") as f:
            finding_concept_map = json.loads(f.read())
        print(f"Loaded map from gs://{MAP_GCS_PATH} ({len(finding_concept_map)} labels)")
    except Exception as e:
        raise RuntimeError(
            f"finding_concept_map.json not found in /content or GCS ({e}). "
            "Re-run Phase A (Cells A1-A3).")

# Drive (best-effort, for Cell B12)
try:
    from google.colab import drive
    drive.mount('/content/drive'); DRIVE_MOUNTED = True
except Exception as e:
    print(f"[warn] Drive not mounted ({e})."); DRIVE_MOUNTED = False

samples = pd.read_csv(SAMPLES_CSV)
rule_rows = []
with open(RULE_JSONL) as f:
    for line in f:
        line = line.strip()
        if line:
            try: rule_rows.append(json.loads(line))
            except Exception: pass
rule_df = pd.DataFrame(rule_rows)[["sample_id","uncertainty_label"]].rename(
    columns={"uncertainty_label":"uncertainty_label_rule"})
samples = samples.rename(columns={"uncertainty_label":"uncertainty_label_medgemma"})
samples = samples.merge(rule_df, on="sample_id", how="left")

# Apply the MedGemma map
samples["rosalia_target"] = samples["finding_label"].map(finding_concept_map)
samples.loc[~samples["rosalia_target"].isin(ROSALIA_LESIONS), "rosalia_target"] = None

n_total = len(samples); n_mapped = samples["rosalia_target"].notna().sum()
print(f"\nMapped to ROSALIA vocab: {n_mapped}/{n_total} ({100*n_mapped/n_total:.1f}%)")
print("Dropped finding_labels (MedGemma said 'none'), top 15:")
print(samples[samples["rosalia_target"].isna()].finding_label.value_counts().head(15).to_string())

samples = samples[samples["rosalia_target"].notna()].reset_index(drop=True)
print(f"\nWorking set: {len(samples)} rows")
print("\nROSALIA target x rule uncertainty:")
print(pd.crosstab(samples["rosalia_target"], samples["uncertainty_label_rule"], margins=True))

def _build_instruction(target, mode=INSTRUCTION_MODE, location=None):
    if mode == "global":
        return INSTRUCTION_TEMPLATE_G.format(target=target)
    loc = location or "right lung"
    return INSTRUCTION_TEMPLATE_B.format(target=target, location=loc)
samples["rosalia_instruction"] = samples["rosalia_target"].apply(_build_instruction)

samples.head(5)[["sample_id","image_id","finding_label","rosalia_target",
                 "rosalia_instruction","uncertainty_label_rule"]]


## Cell B5 — GCS streaming image loader

Streams one PadChest-GR PNG, percentile-windows 16-bit → 8-bit RGB, returns a
PIL image. Cell B6 converts PIL → RGB uint8 numpy for ROSALIA.


In [ ]:
import io
import numpy as np
from PIL import Image

def _window_to_uint8(arr, low_pct=1.0, high_pct=99.0):
    arr = arr.astype(np.float32)
    lo, hi = np.percentile(arr, [low_pct, high_pct])
    if hi - lo < 1.0: hi = lo + 1.0
    return (np.clip((arr - lo)/(hi - lo), 0, 1) * 255).astype(np.uint8)

def load_image_gcs(image_id):
    with fs.open(f"{BUCKET}/{IMG_PREFIX}/{image_id}", "rb") as f:
        data = f.read()
    arr = np.array(Image.open(io.BytesIO(data)))
    if arr.ndim == 3: arr = arr[..., 0]
    return Image.fromarray(_window_to_uint8(arr)).convert("RGB")

_img = load_image_gcs(samples.iloc[0]["image_id"]); _a = np.array(_img)
print(f"OK — {samples.iloc[0]['image_id']}: size={_img.size}, "
      f"min={_a.min()}, max={_a.max()}, mean={_a.mean():.1f}")


## Cell B6 — load ROSALIA (LISA-7B + SAM-H)

Imports the custom `LISAForCausalLM` from the cloned repo, loads
`checkone/ROSALIA-7B-v1` (public, ungated, ~15 GB bf16) with the LISA-7B
tokenizer + CLIP-ViT-L/14 vision tower, and defines
`segment_image_rosalia(...) -> (binary_mask, text_output)`.


In [ ]:
import sys, os
sys.path.insert(0, "/content/rosalia_repo")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import cv2, numpy as np, torch
import torch.nn.functional as F
import transformers
from transformers import CLIPImageProcessor
from model.LISA import LISAForCausalLM
from model.llava import conversation as conversation_lib
from model.llava.mm_utils import tokenizer_image_token
from model.segment_anything.utils.transforms import ResizeLongestSide
from utils.utils import (DEFAULT_IM_END_TOKEN, DEFAULT_IM_START_TOKEN,
                         DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    raise RuntimeError("ROSALIA requires a CUDA GPU (A100/L4).")
IMAGE_SIZE = 1024

def _sam_preprocess(x, img_size=IMAGE_SIZE):
    pixel_mean = torch.tensor([123.675,116.28,103.53]).view(-1,1,1)
    pixel_std  = torch.tensor([58.395,57.12,57.375]).view(-1,1,1)
    x = (x - pixel_mean) / pixel_std
    h, w = x.shape[-2:]
    return F.pad(x, (0, img_size - w, 0, img_size - h))

def load_rosalia():
    tokenizer = transformers.AutoTokenizer.from_pretrained(
        LISA_TOKENIZER, cache_dir=None, model_max_length=512,
        padding_side="right", use_fast=False)
    tokenizer.pad_token = tokenizer.unk_token
    seg_token_idx = tokenizer("[SEG]", add_special_tokens=False).input_ids[0]
    dtype = torch.bfloat16
    model = LISAForCausalLM.from_pretrained(
        ROSALIA_REPO, low_cpu_mem_usage=True, vision_tower=CLIP_VISION_TOWER,
        seg_token_idx=seg_token_idx, torch_dtype=dtype)
    model.config.eos_token_id = tokenizer.eos_token_id
    model.config.bos_token_id = tokenizer.bos_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.get_model().initialize_vision_modules(model.get_model().config)
    model.get_model().get_vision_tower().to(dtype=dtype, device=0)
    model = model.bfloat16().cuda().eval()
    clip_processor = CLIPImageProcessor.from_pretrained(model.config.vision_tower)
    transform = ResizeLongestSide(IMAGE_SIZE)
    if torch.cuda.is_available():
        print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    return model, tokenizer, clip_processor, transform

model, tokenizer, clip_processor, transform = load_rosalia()

def segment_image_rosalia(model, tokenizer, clip_processor, transform, pil_image, instruction):
    image_np = np.array(pil_image)
    if image_np.ndim == 2:
        image_np = np.stack([image_np]*3, axis=-1)
    elif image_np.shape[-1] == 4:
        image_np = image_np[..., :3]
    original_size_list = [image_np.shape[:2]]
    conv = conversation_lib.conv_templates["llava_v1"].copy(); conv.messages = []
    prompt = DEFAULT_IMAGE_TOKEN + "\n" + instruction
    prompt = prompt.replace(DEFAULT_IMAGE_TOKEN,
                            DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN)
    conv.append_message(conv.roles[0], prompt); conv.append_message(conv.roles[1], "")
    prompt = conv.get_prompt()
    image_clip = (clip_processor.preprocess(image_np, return_tensors="pt")["pixel_values"][0]
                  .unsqueeze(0).cuda().bfloat16())
    image_resized = transform.apply_image(image_np)
    resize_list = [image_resized.shape[:2]]
    image = (_sam_preprocess(torch.from_numpy(image_resized).permute(2,0,1).contiguous())
             .unsqueeze(0).cuda().bfloat16())
    input_ids = tokenizer_image_token(prompt, tokenizer, return_tensors="pt").unsqueeze(0).cuda()
    with torch.no_grad():
        output_ids, pred_masks = model.evaluate(
            image_clip, image, input_ids, resize_list, original_size_list,
            max_new_tokens=256, tokenizer=tokenizer)
    out_ids = output_ids[0][output_ids[0] != IMAGE_TOKEN_INDEX]
    text_out = tokenizer.decode(out_ids, skip_special_tokens=False)
    text_out = text_out.replace("\n","").replace("  "," ").split("ASSISTANT:")[-1].split("</s>")[0].strip()
    if len(pred_masks) == 0 or pred_masks[0].numel() == 0:
        mask = np.zeros(original_size_list[0], dtype=np.uint8)
    else:
        mask = (pred_masks[0] > 0).detach().cpu().numpy().astype(np.uint8)
        if mask.ndim == 3: mask = mask[0]
    return mask, text_out

_row = samples.iloc[0]; _img = load_image_gcs(_row.image_id)
_m, _t = segment_image_rosalia(model, tokenizer, clip_processor, transform, _img, _row.rosalia_instruction)
print(f"Smoke: instr={_row.rosalia_instruction!r} mask_sum={int(_m.sum())} text={_t!r}")


## Cell B7 — IoU helpers (mirror project/src/compute_iou.py)

In [ ]:
import numpy as np
from itertools import combinations

def boxes_to_mask(boxes, grid_size=MASK_GRID):
    mask = np.zeros((grid_size, grid_size), dtype=np.uint8)
    if not boxes: return mask
    for x1,y1,x2,y2 in boxes:
        px1=max(0,min(grid_size,int(np.floor(x1*grid_size)))); py1=max(0,min(grid_size,int(np.floor(y1*grid_size))))
        px2=max(0,min(grid_size,int(np.ceil(x2*grid_size))));  py2=max(0,min(grid_size,int(np.ceil(y2*grid_size))))
        if px2>px1 and py2>py1: mask[py1:py2, px1:px2]=1
    return mask

def mask_iou(m1, m2):
    inter=int(np.logical_and(m1,m2).sum()); union=int(np.logical_or(m1,m2).sum())
    return inter/union if union>0 else float("nan")

def boxes_to_intersection_mask(boxes, grid_size=MASK_GRID):
    if not boxes: return np.zeros((grid_size,grid_size),np.uint8)
    masks=[boxes_to_mask([b],grid_size) for b in boxes]; out=masks[0].copy()
    for m in masks[1:]: out=np.logical_and(out,m).astype(np.uint8)
    return out

def boxes_to_outer_bbox_mask(boxes, grid_size=MASK_GRID):
    if not boxes: return np.zeros((grid_size,grid_size),np.uint8)
    xs1=[b[0] for b in boxes]; ys1=[b[1] for b in boxes]; xs2=[b[2] for b in boxes]; ys2=[b[3] for b in boxes]
    return boxes_to_mask([[min(xs1),min(ys1),max(xs2),max(ys2)]], grid_size)

def pairwise_box_ious(boxes, grid_size=MASK_GRID):
    if not boxes or len(boxes)<2: return []
    masks=[boxes_to_mask([b],grid_size) for b in boxes]
    return [mask_iou(masks[i],masks[j]) for i,j in combinations(range(len(masks)),2)]

def pred_probs_to_grid_mask(probs, grid_size=MASK_GRID, threshold=MASK_THRESHOLD):
    arr=np.asarray(probs)
    if arr.ndim==4 and arr.shape[0]==1: arr=arr[0]
    if arr.ndim==3: arr=arr.max(axis=0)
    if arr.ndim!=2: raise ValueError(f"Bad shape {probs.shape}")
    try:
        from skimage.transform import resize
        resized=resize(arr.astype(np.float32),(grid_size,grid_size),order=1,mode="edge",
                       anti_aliasing=False,preserve_range=True)
    except Exception:
        h,w=arr.shape
        ys=np.linspace(0,h-1,grid_size).astype(np.int64); xs=np.linspace(0,w-1,grid_size).astype(np.int64)
        resized=arr[ys[:,None],xs[None,:]]
    return (resized>threshold).astype(np.uint8)

def compute_iou_bundle(pred_probs, r1_boxes, r2_boxes, grid_size=MASK_GRID, threshold=MASK_THRESHOLD):
    pm=pred_probs_to_grid_mask(pred_probs,grid_size,threshold)
    all_boxes=(r1_boxes or [])+(r2_boxes or [])
    per_box=[mask_iou(pm,boxes_to_mask([b],grid_size)) for b in all_boxes]
    return {
        "per_box_ious":per_box,
        "max_per_box_iou":float(np.max(per_box)) if per_box else float("nan"),
        "mean_per_box_iou":float(np.mean(per_box)) if per_box else float("nan"),
        "iou_with_pixel_or_union":mask_iou(pm,boxes_to_mask(all_boxes,grid_size)),
        "iou_with_outer_bbox_union":mask_iou(pm,boxes_to_outer_bbox_mask(all_boxes,grid_size)),
        "iou_with_intersection":mask_iou(pm,boxes_to_intersection_mask(all_boxes,grid_size)),
        "iou_with_reader1_union":mask_iou(pm,boxes_to_mask(r1_boxes or [],grid_size)),
        "iou_with_reader2_union":mask_iou(pm,boxes_to_mask(r2_boxes or [],grid_size)),
        "reader_iou_union":mask_iou(boxes_to_mask(r1_boxes or [],grid_size),boxes_to_mask(r2_boxes or [],grid_size)),
        "pairwise_box_ious":pairwise_box_ious(all_boxes,grid_size),
        "mean_pairwise_iou":float(np.mean(pairwise_box_ious(all_boxes,grid_size))) if len(all_boxes)>=2 else float("nan"),
        "pred_area":int(pm.sum()),
        "num_boxes_total":len(all_boxes),
        "num_reader1_boxes":len(r1_boxes or []),
        "num_reader2_boxes":len(r2_boxes or []),
    }, pm


## Cell B8 — SMOKE TEST (50 samples)

In [ ]:
import random, json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt, matplotlib.patches as patches
random.seed(SMOKE_SEED)

def stratified_sample(df, n_per_class):
    out=[]
    for cls in ["certain","uncertain"]:
        sub=df[df["uncertainty_label_rule"]==cls]
        out.append(sub.sample(n=min(n_per_class,len(sub)), random_state=SMOKE_SEED))
    return pd.concat(out).sample(frac=1, random_state=SMOKE_SEED).reset_index(drop=True)

smoke = stratified_sample(samples, SMOKE_N//2)
print(f"Smoke set: {len(smoke)}")
smoke_rows=[]; sample_masks={}
for row in tqdm(smoke.itertuples(), total=len(smoke), desc="smoke"):
    img=load_image_gcs(row.image_id)
    mask,text_out=segment_image_rosalia(model,tokenizer,clip_processor,transform,img,row.rosalia_instruction)
    r1=json.loads(row.reader1_boxes) if isinstance(row.reader1_boxes,str) else row.reader1_boxes
    r2=json.loads(row.reader2_boxes) if isinstance(row.reader2_boxes,str) else row.reader2_boxes
    bundle,pm=compute_iou_bundle(mask,r1,r2)
    smoke_rows.append({"sample_id":row.sample_id,"image_id":row.image_id,
        "finding_label":row.finding_label,"rosalia_target":row.rosalia_target,
        "rosalia_instruction":row.rosalia_instruction,"rosalia_text_output":text_out,
        "uncertainty_label_rule":row.uncertainty_label_rule,
        "uncertainty_label_medgemma":row.uncertainty_label_medgemma,
        "sentence_raw":row.sentence, **bundle})
    sample_masks[row.sample_id]=pm

smoke_df=pd.DataFrame(smoke_rows)
display(smoke_df[["sample_id","uncertainty_label_rule","rosalia_target",
    "rosalia_text_output","iou_with_pixel_or_union","iou_with_outer_bbox_union",
    "mean_per_box_iou","reader_iou_union","num_boxes_total"]].head(5))

fig,axes=plt.subplots(5,4,figsize=(16,20))
for i,row in enumerate(smoke_df.head(5).itertuples()):
    img=load_image_gcs(row.image_id); iw,ih=img.size
    r1=json.loads(samples.loc[samples.sample_id==row.sample_id,"reader1_boxes"].iloc[0])
    r2=json.loads(samples.loc[samples.sample_id==row.sample_id,"reader2_boxes"].iloc[0])
    pm=sample_masks[row.sample_id]
    axes[i,0].imshow(img,cmap="gray"); axes[i,0].set_title("image"); axes[i,0].axis("off")
    axes[i,1].imshow(img,cmap="gray")
    for b,c in [(r1,"red"),(r2,"blue")]:
        for x1,y1,x2,y2 in b:
            axes[i,1].add_patch(patches.Rectangle((x1*iw,y1*ih),(x2-x1)*iw,(y2-y1)*ih,lw=2,edgecolor=c,facecolor="none"))
    axes[i,1].set_title("GT (R1=red,R2=blue)"); axes[i,1].axis("off")
    axes[i,2].imshow(pm,cmap="gray"); axes[i,2].set_title(f"pred IoU(or)={row.iou_with_pixel_or_union:.2f}"); axes[i,2].axis("off")
    axes[i,3].imshow(img,cmap="gray")
    pm_disp=np.array(Image.fromarray((pm*255).astype(np.uint8)).resize((iw,ih),Image.NEAREST))
    axes[i,3].imshow(np.ma.masked_where(pm_disp==0,pm_disp),cmap="Reds",alpha=0.5)
    axes[i,3].set_title(f"[{row.uncertainty_label_rule}] {row.rosalia_target}\n{row.rosalia_text_output[:60]}"); axes[i,3].axis("off")
plt.tight_layout(); plt.savefig("/content/rosalia_smoke_grid.png",dpi=110); plt.show()


## Cell B9 — FULL RUN (resumable)

In [ ]:
import io, json
import numpy as np
from tqdm.auto import tqdm

def exists_in_bucket(path):
    try: return fs.exists(path)
    except Exception: return False
def upload_json(path,obj):
    with fs.open(path,"w") as f: f.write(json.dumps(obj))
def upload_npz(path,mask):
    buf=io.BytesIO(); np.savez_compressed(buf,mask=mask); buf.seek(0)
    with fs.open(path,"wb") as f: f.write(buf.read())

all_records=[]; n_skipped=0; n_new=0
for row in tqdm(samples.itertuples(), total=len(samples), desc="full"):
    out_json=f"{BUCKET}/{OUT_PREFIX}/{row.sample_id}.json"
    out_npz =f"{BUCKET}/{MASK_PREFIX}/{row.sample_id}.npz"
    if exists_in_bucket(out_json):
        with fs.open(out_json,"r") as f: all_records.append(json.loads(f.read()))
        n_skipped+=1; continue
    try:
        img=load_image_gcs(row.image_id)
        mask,text_out=segment_image_rosalia(model,tokenizer,clip_processor,transform,img,row.rosalia_instruction)
        r1=json.loads(row.reader1_boxes) if isinstance(row.reader1_boxes,str) else row.reader1_boxes
        r2=json.loads(row.reader2_boxes) if isinstance(row.reader2_boxes,str) else row.reader2_boxes
        bundle,pm=compute_iou_bundle(mask,r1,r2)
        rec={"sample_id":row.sample_id,"image_id":row.image_id,"finding_label":row.finding_label,
             "rosalia_target":row.rosalia_target,"rosalia_instruction":row.rosalia_instruction,
             "rosalia_text_output":text_out,"uncertainty_label_rule":row.uncertainty_label_rule,
             "uncertainty_label_medgemma":row.uncertainty_label_medgemma,"sentence_raw":row.sentence,**bundle}
        upload_json(out_json,rec); upload_npz(out_npz,pm); all_records.append(rec); n_new+=1
    except Exception as e:
        print(f"[skip] {row.sample_id}: {type(e).__name__}: {e}"); continue

results_df=pd.DataFrame(all_records)
print(f"\nDone. skipped={n_skipped} new={n_new} total={len(results_df)}")
results_df.to_csv("/content/rosalia_per_sample_ious.csv", index=False)


## Cell B10 — aggregates stratified by rule-based uncertainty

In [ ]:
import json, numpy as np
RNG=np.random.default_rng(42); BOOT_N=2000
IOU_COLS=["iou_with_pixel_or_union","iou_with_outer_bbox_union","iou_with_intersection",
          "iou_with_reader1_union","iou_with_reader2_union","reader_iou_union",
          "mean_per_box_iou","max_per_box_iou","mean_pairwise_iou"]
for c in IOU_COLS: results_df[c]=pd.to_numeric(results_df[c],errors="coerce")

print("=== By uncertainty group ===")
print(results_df.groupby("uncertainty_label_rule")[IOU_COLS].agg(["count","mean","median","std"]))
print("\n=== By ROSALIA target x uncertainty ===")
pt=results_df.groupby(["rosalia_target","uncertainty_label_rule"])[IOU_COLS].agg(["count","mean","median"])
print(pt); pt.to_csv("/content/rosalia_per_target_ious.csv")
results_df.groupby(["uncertainty_label_rule","finding_label"])[IOU_COLS].agg(
    ["count","mean","median"]).to_csv("/content/rosalia_per_finding_ious.csv")

def boot_mean_ci(x,n=BOOT_N,alpha=0.05):
    x=np.asarray(x,dtype=float); x=x[~np.isnan(x)]
    if len(x)==0: return (float("nan"),)*3
    idx=RNG.integers(0,len(x),size=(n,len(x))); means=x[idx].mean(axis=1)
    lo,hi=np.quantile(means,[alpha/2,1-alpha/2]); return float(means.mean()),float(lo),float(hi)

agg_json={}
for cls,sub in results_df.groupby("uncertainty_label_rule"):
    mean,lo,hi=boot_mean_ci(sub["iou_with_pixel_or_union"])
    agg_json[str(cls)]={"n":int(len(sub)),"iou_with_pixel_or_union_mean":mean,
        "iou_with_pixel_or_union_ci95":[lo,hi],
        "iou_with_outer_bbox_union_mean":float(sub["iou_with_outer_bbox_union"].mean()),
        "iou_with_intersection_mean":float(sub["iou_with_intersection"].mean()),
        "reader_iou_union_mean":float(sub["reader_iou_union"].mean()),
        "mean_per_box_iou_mean":float(sub["mean_per_box_iou"].mean())}
json.dump(agg_json, open("/content/rosalia_aggregate_ious.json","w"), indent=2)
agg_json


## Cell B11 — plots

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
for cls,color in [("certain","tab:blue"),("uncertain","tab:orange")]:
    sub=results_df[results_df["uncertainty_label_rule"]==cls]["iou_with_pixel_or_union"].dropna()
    plt.hist(sub,bins=40,alpha=0.55,label=f"{cls} (n={len(sub)})",color=color)
plt.xlabel("IoU(pred, pixel-OR union of GT boxes)"); plt.ylabel("Count")
plt.title("ROSALIA IoU by rule-based spatial uncertainty"); plt.legend(); plt.tight_layout()
plt.savefig("/content/rosalia_iou_histogram_by_group.png",dpi=120); plt.show()

plot_cols=["iou_with_pixel_or_union","iou_with_outer_bbox_union","iou_with_intersection",
           "mean_per_box_iou","max_per_box_iou","reader_iou_union"]
means=results_df.groupby("uncertainty_label_rule")[plot_cols].mean()
fig,ax=plt.subplots(figsize=(10,4)); means.T.plot(kind="bar",ax=ax)
ax.set_ylabel("Mean IoU"); ax.set_title("ROSALIA mean IoU per metric, by uncertainty group")
ax.set_xticklabels(plot_cols,rotation=30,ha="right"); plt.tight_layout()
plt.savefig("/content/rosalia_iou_by_uncertainty_group.png",dpi=120); plt.show()

plt.figure(figsize=(6,5))
for cls,color in [("certain","tab:blue"),("uncertain","tab:orange")]:
    sub=results_df[results_df["uncertainty_label_rule"]==cls]
    plt.scatter(sub["reader_iou_union"],sub["iou_with_pixel_or_union"],s=10,alpha=0.5,label=cls,color=color)
plt.xlabel("Reader-Reader IoU (agreement)"); plt.ylabel("ROSALIA IoU")
plt.title("ROSALIA quality vs annotator agreement"); plt.legend(); plt.tight_layout()
plt.savefig("/content/rosalia_agreement_vs_iou_scatter.png",dpi=120); plt.show()


## Cell B12 — push aggregated results back to the repo via Drive

In [ ]:
import shutil, os
if globals().get("DRIVE_MOUNTED",False) and os.path.isdir("/content/drive/MyDrive"):
    out_dir=f"{DRIVE_REPO_ROOT}/project/outputs"; fig_dir=f"{DRIVE_REPO_ROOT}/project/figures"
else:
    out_dir="/content/outputs"; fig_dir="/content/figures"
    print("Drive not mounted — writing to /content/.")
os.makedirs(out_dir,exist_ok=True); os.makedirs(fig_dir,exist_ok=True)
for fn in ("rosalia_per_sample_ious.csv","rosalia_per_finding_ious.csv",
           "rosalia_per_target_ious.csv","rosalia_aggregate_ious.json",
           "finding_concept_map.json"):
    if os.path.exists(f"/content/{fn}"): shutil.copy(f"/content/{fn}", f"{out_dir}/{fn}")
for fn in ("rosalia_iou_histogram_by_group.png","rosalia_iou_by_uncertainty_group.png",
           "rosalia_agreement_vs_iou_scatter.png","rosalia_smoke_grid.png"):
    if os.path.exists(f"/content/{fn}"): shutil.copy(f"/content/{fn}", f"{fig_dir}/{fn}")
print(f"Done. outputs -> {out_dir}, figures -> {fig_dir}")
